In [1]:
!git clone https://github.com/Mickrbl/TaxiNY.git
%cd TaxiNY

Cloning into 'TaxiNY'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 51 (delta 1), reused 4 (delta 1), pack-reused 46 (from 1)
Receiving objects: 100% (51/51), 791.63 MiB | 46.20 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Updating files: 100% (32/32), done.
/kaggle/working/TaxiNY


In [2]:
# Importiamo le librerie necessarie
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime

# Assicurati di avere pyarrow installato (pip install pyarrow)

In [3]:
# Definiamo la cartella dove salverai tutti i file Parquet (es. da gennaio a dicembre)
# Esempio: 'yellow_tripdata_2025-01.parquet', 'yellow_tripdata_2025-02.parquet', ecc.
DATA_DIR = '/kaggle/working/TaxiNY/data/taxi_gialli' 

# Trova tutti i file parquet nella cartella
all_files = glob.glob(os.path.join(DATA_DIR, "yellow_tripdata_*.parquet"))

print(f"Trovati {len(all_files)} file da processare.")

# Leggiamo tutti i file e li uniamo in un unico DataFrame
# Usiamo le colonne strettamente necessarie per risparmiare RAM
columns_to_keep = [
    'tpep_pickup_datetime', 
    'tpep_dropoff_datetime', 
    'PULocationID', 
    'DOLocationID'
]

df_list = []
for file in all_files:
    print(f"Caricamento di {file}...")
    df_temp = pd.read_parquet(file, columns=columns_to_keep, engine='pyarrow')
    df_list.append(df_temp)

# Concateniamo tutto l'anno
df = pd.concat(df_list, ignore_index=True)
print(f"Dataset completo caricato. Numero totale di righe: {len(df)}")

Trovati 12 file da processare.
Caricamento di /kaggle/working/TaxiNY/data/taxi_gialli/yellow_tripdata_2025-09.parquet...
Caricamento di /kaggle/working/TaxiNY/data/taxi_gialli/yellow_tripdata_2025-10.parquet...
Caricamento di /kaggle/working/TaxiNY/data/taxi_gialli/yellow_tripdata_2025-11.parquet...
Caricamento di /kaggle/working/TaxiNY/data/taxi_gialli/yellow_tripdata_2025-08.parquet...
Caricamento di /kaggle/working/TaxiNY/data/taxi_gialli/yellow_tripdata_2025-07.parquet...
Caricamento di /kaggle/working/TaxiNY/data/taxi_gialli/yellow_tripdata_2025-01.parquet...
Caricamento di /kaggle/working/TaxiNY/data/taxi_gialli/yellow_tripdata_2025-03.parquet...
Caricamento di /kaggle/working/TaxiNY/data/taxi_gialli/yellow_tripdata_2025-02.parquet...
Caricamento di /kaggle/working/TaxiNY/data/taxi_gialli/yellow_tripdata_2025-04.parquet...
Caricamento di /kaggle/working/TaxiNY/data/taxi_gialli/yellow_tripdata_2025-12.parquet...
Caricamento di /kaggle/working/TaxiNY/data/taxi_gialli/yellow_tripdat

In [4]:
TARGET_YEAR = 2025

# Rimuoviamo i valori nulli nelle zone o nelle date
df = df.dropna(subset=['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID'])

# Filtriamo solo i viaggi che avvengono effettivamente nel TARGET_YEAR
df = df[
    (df['tpep_pickup_datetime'].dt.year == TARGET_YEAR) & 
    (df['tpep_dropoff_datetime'].dt.year == TARGET_YEAR)
]

# Rimuoviamo viaggi con durata negativa o nulla
df = df[df['tpep_dropoff_datetime'] > df['tpep_pickup_datetime']]

print(f"Righe rimanenti dopo la pulizia: {len(df)}")

Righe rimanenti dopo la pulizia: 48175495


In [5]:
# Definiamo l'ordine corretto dei giorni della settimana
giorni_ordinati = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# ---------------------------------------------------------
# Estraiamo le features per i Pick-ups (Domanda)
# ---------------------------------------------------------
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour

# Usiamo .dt.day_name() invece di .dt.dayofweek per avere i nomi in inglese
df['pickup_day_of_week'] = df['tpep_pickup_datetime'].dt.day_name()

# Trasformiamo la colonna in una categoria ordinata
df['pickup_day_of_week'] = pd.Categorical(df['pickup_day_of_week'], categories=giorni_ordinati, ordered=True)

# La logica del weekend si può aggiornare basandosi sul nome del giorno
df['pickup_is_weekend'] = df['pickup_day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# ---------------------------------------------------------
# Estraiamo le features per i Drop-offs (Offerta/Disponibilità)
# ---------------------------------------------------------
df['dropoff_hour'] = df['tpep_dropoff_datetime'].dt.hour

# Stessa cosa per i drop-off
df['dropoff_day_of_week'] = df['tpep_dropoff_datetime'].dt.day_name()
df['dropoff_day_of_week'] = pd.Categorical(df['dropoff_day_of_week'], categories=giorni_ordinati, ordered=True)

In [6]:
# Raggruppiamo per Zona di partenza, Giorno e Ora
demand_df = df.groupby(['PULocationID', 'pickup_day_of_week', 'pickup_hour']).size().reset_index(name='predicted_demand')

# Rinominiamo le colonne per renderle uniformi prima del merge
demand_df = demand_df.rename(columns={
    'PULocationID': 'zone_id',
    'pickup_day_of_week': 'day_of_week',
    'pickup_hour': 'hour'
})

print("Aggregazione Domanda completata. Anteprima:")
display(demand_df.head())

/tmp/ipykernel_55/3346842740.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  demand_df = df.groupby(['PULocationID', 'pickup_day_of_week', 'pickup_hour']).size().reset_index(name='predicted_demand')


Aggregazione Domanda completata. Anteprima:


,zone_id,day_of_week,hour,predicted_demand
0,1,Monday,0,9
1,1,Monday,1,8
2,1,Monday,2,5
3,1,Monday,3,5
4,1,Monday,4,18


In [7]:
# Raggruppiamo per Zona di arrivo, Giorno e Ora
supply_df = df.groupby(['DOLocationID', 'dropoff_day_of_week', 'dropoff_hour']).size().reset_index(name='predicted_supply')

# Rinominiamo le colonne
supply_df = supply_df.rename(columns={
    'DOLocationID': 'zone_id',
    'dropoff_day_of_week': 'day_of_week',
    'dropoff_hour': 'hour'
})

print("Aggregazione Offerta completata. Anteprima:")
display(supply_df.head())

/tmp/ipykernel_55/774326218.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  supply_df = df.groupby(['DOLocationID', 'dropoff_day_of_week', 'dropoff_hour']).size().reset_index(name='predicted_supply')


Aggregazione Offerta completata. Anteprima:


,zone_id,day_of_week,hour,predicted_supply
0,1,Monday,0,19
1,1,Monday,1,20
2,1,Monday,2,28
3,1,Monday,3,96
4,1,Monday,4,278


In [8]:
# Uniamo i due DataFrame basandoci su zona, giorno e ora (Outer Join per non perdere dati)
final_dataset = pd.merge(demand_df, supply_df, on=['zone_id', 'day_of_week', 'hour'], how='outer')

# Se in una determinata ora ci sono drop-off ma non pick-up (o viceversa), avremo dei NaN.
# Riempiamo questi vuoti con 0 (es. 0 domanda o 0 taxi in arrivo)
final_dataset['predicted_demand'] = final_dataset['predicted_demand'].fillna(0).astype(int)
final_dataset['predicted_supply'] = final_dataset['predicted_supply'].fillna(0).astype(int)


# Riapplichiamo la logica del weekend cercando i nomi esatti dei giorni
final_dataset['is_weekend'] = final_dataset['day_of_week'].isin(['Saturday', 'Sunday']).astype(int)

# Ordiniamo il dataset per zona, giorno e ora
final_dataset = final_dataset.sort_values(by=['zone_id', 'day_of_week', 'hour']).reset_index(drop=True)

print("Dataset finale pronto per l'addestramento!")
display(final_dataset.head())

# Salviamo il risultato finale, pronto per lo Step 3 (Machine Learning)
# Lo salviamo in parquet per mantenere le performance e i tipi di dato
final_dataset.to_parquet('nyc_taxi_aggregated_2025.parquet', engine='pyarrow', index=False)

Dataset finale pronto per l'addestramento!


,zone_id,day_of_week,hour,predicted_demand,predicted_supply,is_weekend
0,1,Monday,0,9,19,0
1,1,Monday,1,8,20,0
2,1,Monday,2,5,28,0
3,1,Monday,3,5,96,0
4,1,Monday,4,18,278,0


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

# Assicuriamoci che 'day_of_week' sia di tipo category (lo avevamo già fatto, ma è meglio ribadirlo per sicurezza)
final_dataset['day_of_week'] = final_dataset['day_of_week'].astype('category')
final_dataset['zone_id'] = final_dataset['zone_id'].astype('category')

# Definiamo le features (Input) e i due Target (Output)
X = final_dataset[['zone_id', 'day_of_week', 'hour', 'is_weekend']]
y_demand = final_dataset['predicted_demand']
y_supply = final_dataset['predicted_supply']

# Eseguiamo il Train/Test Split (usiamo l'80% per il training, 20% per il test)
# Passiamo sia y_demand che y_supply per splittare tutto contemporaneamente
X_train, X_test, y_demand_train, y_demand_test, y_supply_train, y_supply_test = train_test_split(
    X, y_demand, y_supply, test_size=0.2, random_state=42
)

print(f"Dimensioni Training set: {X_train.shape[0]} righe")
print(f"Dimensioni Test set: {X_test.shape[0]} righe")

Dimensioni Training set: 35347 righe
Dimensioni Test set: 8837 righe


In [10]:
print("Inizio addestramento dei modelli XGBoost...")

# Parametri base per il nostro regressore. 
# tree_method='hist' è obbligatorio quando si usa enable_categorical=True
xgb_params = {
    'n_estimators': 200,          # Numero di alberi decisionali
    'learning_rate': 0.1,         # Tasso di apprendimento
    'max_depth': 8,               # Profondità massima degli alberi (importante per le interazioni complesse zona/ora)
    'enable_categorical': True,   # Permette a XGBoost di leggere 'Monday', 'Tuesday', ecc.
    'tree_method': 'hist',        # Ottimizzato e necessario per le categorie
    'random_state': 42,
    'device': 'cuda',
    'n_jobs': -1                  # Usa tutti i core del processore per velocizzare
}

# 1. Modello per la DOMANDA
model_demand = xgb.XGBRegressor(**xgb_params)
model_demand.fit(X_train, y_demand_train)
print("Modello Domanda (Pick-ups) addestrato!")

# 2. Modello per l'OFFERTA
model_supply = xgb.XGBRegressor(**xgb_params)
model_supply.fit(X_train, y_supply_train)
print("Modello Offerta (Drop-offs) addestrato!")

Inizio addestramento dei modelli XGBoost...
Modello Domanda (Pick-ups) addestrato!
Modello Offerta (Drop-offs) addestrato!


In [11]:
# Facciamo le predizioni sui dati di test
preds_demand = model_demand.predict(X_test)
preds_supply = model_supply.predict(X_test)

# Dato che non possono esistere passeggeri o taxi negativi, convertiamo eventuali predizioni negative a 0
preds_demand = [max(0, p) for p in preds_demand]
preds_supply = [max(0, p) for p in preds_supply]

# Calcoliamo il MAE (Mean Absolute Error)
# Ci dice di quanti "viaggi" o "taxi" stiamo sbagliando in media per ogni zona/ora
mae_demand = mean_absolute_error(y_demand_test, preds_demand)
mae_supply = mean_absolute_error(y_supply_test, preds_supply)

print("--- VALUTAZIONE DEI MODELLI ---")
print(f"Errore Medio Assoluto (MAE) sulla Domanda: {mae_demand:.2f} passeggeri/ora")
print(f"Errore Medio Assoluto (MAE) sull'Offerta: {mae_supply:.2f} taxi/ora")

# Mostriamo una piccola anteprima delle predizioni a confronto con i valori reali
comparison_df = X_test.copy()
comparison_df['Real_Demand'] = y_demand_test
comparison_df['Pred_Demand'] = [int(p) for p in preds_demand]
comparison_df['Real_Supply'] = y_supply_test
comparison_df['Pred_Supply'] = [int(p) for p in preds_supply]

display(comparison_df.head(10))

--- VALUTAZIONE DEI MODELLI ---
Errore Medio Assoluto (MAE) sulla Domanda: 199.90 passeggeri/ora
Errore Medio Assoluto (MAE) sull'Offerta: 202.41 taxi/ora


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [15:20:20] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


,zone_id,day_of_week,hour,is_weekend,Real_Demand,Pred_Demand,Real_Supply,Pred_Supply
14363,86,Thursday,11,0,80,812,69,1220
4012,24,Sunday,4,1,224,196,581,481
7168,43,Friday,16,0,9304,8644,4965,4373
17947,109,Saturday,19,1,0,144,6,264
7441,45,Wednesday,1,0,188,278,277,248
24751,150,Wednesday,7,0,29,242,56,273
40280,242,Saturday,8,1,51,0,76,0
23357,142,Monday,5,0,846,818,673,802
37839,228,Tuesday,15,0,173,3536,218,4411
16307,98,Monday,11,0,31,79,59,0


In [13]:
print(f"Media reale della domanda: {y_demand_test.mean():.2f}")
print(f"Media reale della supply: {y_supply_test.mean():.2f}")


Media reale della domanda: 1040.93
Media reale della supply: 1058.27
